In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1")
CATALOG = PROJECT_DIR / "data" / "dmdg_catalog_z0.csv"
df = pd.read_csv(CATALOG)

print(f"Total catalog objects: {len(df):,}")
resolved = df[
    (df["N_star"] >= 500) &
    (df["SubhaloFlag"] == 1) &
    (df["f_DM"].notna())
].copy()

print(f"Resolved galaxies: {len(resolved):,}")
centrals = resolved[
    resolved["IsSatellite"] == False
].copy()
dmdgs = centrals[
    centrals["f_DM"] < 0.5
].copy()
dmdgs_non_extreme = dmdgs[
   dmdgs["f_DM"] > 0.05
].copy()
print(f"Centrals: {len(centrals):,}")
print(f"DMDGS (total): {len(dmdgs):,}")
print(f"DMDGS (non-tidal stripping): {len(dmdgs_non_extreme):,}")

Total catalog objects: 13,922,457
Resolved galaxies: 154,638
Centrals: 93,039
DMDGS (total): 96
DMDGS (non-tidal stripping): 96


In [8]:

features = ["M_total_2Rh", "f_DM", "M_star"]
z_score_cols = []

for col in features:

    median_val = dmdgs[col].median()

    mad_val = (
        dmdgs[col] - median_val
    ).abs().median()

    if mad_val == 0:
        mad_val = dmdgs[col].std()

    z_col = f"{col}_z"

    dmdgs[z_col] = (
        (dmdgs[col] - median_val)
        / mad_val
    ).abs()

    z_score_cols.append(z_col)
dmdgs["total_distance"] = np.sqrt(
    (
        dmdgs[z_score_cols] ** 2
    ).sum(axis=1)
)

dmdgs_z_sorted = dmdgs.sort_values(
    by="total_distance"
)

representative = dmdgs_z_sorted.head(5)
print()
print("=" * 70)
print("FIVE REPRESENTATIVE DMDGs")
print("=" * 70)

print(
    representative[
        [
            "SubhaloID",
            "GroupID",
            "M_star",
            "M_total_2Rh",
            "f_DM",
            "R_half_star",
            "total_distance"
        ]
    ].to_string(index=False)
)
output_file = PROJECT_DIR / "data" / "representative_dmdgs.csv"

representative.to_csv(
    output_file,
    index=False
)

print()
print(f"Saved to:")
print(output_file)


FIVE REPRESENTATIVE DMDGs
 SubhaloID  GroupID   M_star  M_total_2Rh     f_DM  R_half_star  total_distance
   1603468    17403 2.442713     5.182994 0.488034     2.083286        0.428917
   1784604    30047 2.662195     5.193266 0.486302     2.703830        0.453012
   1763757    28223 2.175894     4.400174 0.489761     2.314904        0.465986
   1804523    31931 2.099484     4.272826 0.483848     2.035346        0.479435
   1723072    24978 2.805873     4.361835 0.489500     2.031300        0.503823

Saved to:
C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1\data\representative_dmdgs.csv
